In [98]:
import os
import cv2
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from skimage.feature import hog

import matplotlib.pyplot as plt
import pandas as pd

DATASET_PATH = 'UCMerced_LandUse/Images' 
TEST_SIZE = 0.2
HIST_BINS = 256
HOG_PARAMS = {
    'orientations': 8,
    'pixels_per_cell': (64, 64),
    'cells_per_block': (2, 2),
    'block_norm': 'L2-Hys'
}

In [99]:
def extract_histogram(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hist, _ = np.histogram(gray, bins=HIST_BINS, range=(0, 256))
    hist = hist.astype('float32')
    return hist / np.sum(hist)

def extract_hog(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fd, img= hog(gray, **HOG_PARAMS, visualize=True)
    return fd

In [100]:
features = []
labels = []

for label in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, label)
    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            img = cv2.imread(img_path)
            if img is not None:
                feat = extract_histogram(img)
                features.append(feat)
                labels.append(label)

X = np.array(features)   
y = np.array(labels)   

In [101]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [102]:
X_train.shape

(800, 256)

In [103]:
classifiers = []

for n in [10, 50, 100]:
    clf = RandomForestClassifier(n_estimators=n, random_state=42)
    classifiers.append(('RF', f'n={n}', clf))

for k in [3, 10, 20]:
    clf = KNeighborsClassifier(n_neighbors=k)
    classifiers.append(('KNN', f'k={k}', clf))

In [104]:
classifiers

[('RF', 'n=10', RandomForestClassifier(n_estimators=10, random_state=42)),
 ('RF', 'n=50', RandomForestClassifier(n_estimators=50, random_state=42)),
 ('RF', 'n=100', RandomForestClassifier(random_state=42)),
 ('KNN', 'k=3', KNeighborsClassifier(n_neighbors=3)),
 ('KNN', 'k=10', KNeighborsClassifier(n_neighbors=10)),
 ('KNN', 'k=20', KNeighborsClassifier(n_neighbors=20))]

In [105]:
for clf_type, param, clf in classifiers:
    clf.fit(X_train, y_train) #train             

    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test = accuracy_score(y_test, y_pred_test)

    cm_train = confusion_matrix(y_train, y_pred_train)
    cm_test = confusion_matrix(y_test, y_pred_test)

    print(f'{clf_type} ({param})')
    print(f'Acuratete antrenare: {acc_train*100:.2f}')
    print(f'Acuratete test:      {acc_test*100:.2f}')
    print()
    print('Matrice confuzie - antrenare:')
    print(cm_train)
    print()
    print('Matrice confuzie - test:')
    print(cm_test)
    print()

RF (n=10)
Acuratete antrenare: 99.38
Acuratete test:      68.50

Matrice confuzie - antrenare:
[[80  0  0  0  0  0  0  0  0  0]
 [ 0 80  0  0  0  0  0  0  0  0]
 [ 0  1 79  0  0  0  0  0  0  0]
 [ 0  0  0 80  0  0  0  0  0  0]
 [ 0  0  0  0 79  0  1  0  0  0]
 [ 0  0  0  0  0 80  0  0  0  0]
 [ 0  0  0  0  0  0 80  0  0  0]
 [ 1  0  0  0  0  0  0 79  0  0]
 [ 0  0  0  0  0  0  0  0 80  0]
 [ 0  2  0  0  0  0  0  0  0 78]]

Matrice confuzie - test:
[[14  0  0  0  0  4  0  1  0  1]
 [ 0 16  1  0  0  0  3  0  0  0]
 [ 1  1 16  0  1  0  0  0  0  1]
 [ 4  0  1 13  0  0  0  0  0  2]
 [ 0  3  0  0 15  0  1  0  1  0]
 [ 0  1  1  0  0 16  0  1  0  1]
 [ 1  1  0  0  3  0 13  0  2  0]
 [ 1  0  1  0  0  1  0 17  0  0]
 [ 0  4  2  1  0  1  2  0 10  0]
 [ 2  4  1  2  1  0  2  0  1  7]]

RF (n=50)
Acuratete antrenare: 100.00
Acuratete test:      75.00

Matrice confuzie - antrenare:
[[80  0  0  0  0  0  0  0  0  0]
 [ 0 80  0  0  0  0  0  0  0  0]
 [ 0  0 80  0  0  0  0  0  0  0]
 [ 0  0  0 80  0  0  

In [106]:
# part 2 HOG

features = []
labels = []

for label in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, label)
    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            img = cv2.imread(img_path)
            if img.shape[:2] != (256, 256):  # nota: am gasit cel putin o imagine de dimensiuni 253x256, nu e standardizat setul
                img = cv2.resize(img, (256, 256))
            feat = extract_hog(img)
            features.append(feat)
            labels.append(label)

In [107]:
len(features[155]) #imaginea 156 avea probleme

288

In [108]:
X = np.array(features)   
y = np.array(labels)   

In [109]:
X.shape

(1000, 288)

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape

(800, 288)

In [111]:
for clf_type, param, clf in classifiers:
    clf.fit(X_train, y_train) #train             

    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test = accuracy_score(y_test, y_pred_test)

    cm_train = confusion_matrix(y_train, y_pred_train)
    cm_test = confusion_matrix(y_test, y_pred_test)

    print(f'{clf_type} ({param})')
    print(f'Acuratete antrenare: {acc_train*100:.2f}')
    print(f'Acuratete test:      {acc_test*100:.2f}')
    print()
    print('Matrice confuzie - antrenare:')
    print(cm_train)
    print()
    print('Matrice confuzie - test:')
    print(cm_test)
    print()

RF (n=10)
Acuratete antrenare: 99.88
Acuratete test:      61.00

Matrice confuzie - antrenare:
[[80  0  0  0  0  0  0  0  0  0]
 [ 0 80  0  0  0  0  0  0  0  0]
 [ 0  0 80  0  0  0  0  0  0  0]
 [ 0  0  0 80  0  0  0  0  0  0]
 [ 0  0  0  0 80  0  0  0  0  0]
 [ 0  0  0  0  0 80  0  0  0  0]
 [ 0  0  0  0  0  0 80  0  0  0]
 [ 0  1  0  0  0  0  0 79  0  0]
 [ 0  0  0  0  0  0  0  0 80  0]
 [ 0  0  0  0  0  0  0  0  0 80]]

Matrice confuzie - test:
[[13  0  0  0  0  0  0  2  5  0]
 [ 0 12  1  2  2  1  2  0  0  0]
 [ 0  1 17  0  0  0  0  0  0  2]
 [ 0  2  1 14  0  0  0  0  1  2]
 [ 2  4  0  0  7  0  7  0  0  0]
 [ 0  0  0  0  0 17  0  3  0  0]
 [ 0  1  1  1  3  1 13  0  0  0]
 [ 0  0  2  0  0  9  0  9  0  0]
 [ 1  0  1  0  0  0  0  0 18  0]
 [ 0  5  7  5  0  0  1  0  0  2]]

RF (n=50)
Acuratete antrenare: 100.00
Acuratete test:      73.00

Matrice confuzie - antrenare:
[[80  0  0  0  0  0  0  0  0  0]
 [ 0 80  0  0  0  0  0  0  0  0]
 [ 0  0 80  0  0  0  0  0  0  0]
 [ 0  0  0 80  0  0  

In [ ]:
# am incercat cu HOG doua variante, nr. features este invers proportional cu acuratetea, pana la un punct cand devin prea putine
